In [7]:
import os
import sys
from pathlib import Path

# Lock working directory to main project root (2 levels up from src/notebook/)
current_dir = Path.cwd()
if current_dir.name == "notebook" and current_dir.parent.name == "src":
    project_root = current_dir.parent.parent
    os.chdir(project_root)
elif current_dir.name == "src":
    project_root = current_dir.parent
    os.chdir(project_root)
else:
    project_root = current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Kernel working directory locked to:", Path.cwd())

Kernel working directory locked to: /home/s2025ae05884/workspace/heart-disease-predict


In [8]:
%%writefile src/train.py
import os
from pathlib import Path

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
)

import mlflow
import mlflow.sklearn

from src.preprocess import build_preprocessor, ALL_FEATURES, TARGET_COL


Overwriting src/train.py


In [9]:
%%writefile -a src/train.py


def plot_and_save_artifacts(y_true, y_pred, y_prob, model_name, run_id, output_dir="screenshots"):
    """Generate evaluation plots in a unique MLflow-run directory."""
    run_dir = Path(output_dir) / "mlflow_runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix - {model_name}", fontweight="bold")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    cm_path = run_dir / f"cm_{model_name.lower()}.png"
    plt.tight_layout()
    plt.savefig(cm_path, dpi=300)
    plt.close()

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, color="#d95f02", lw=2, label=f"ROC curve (AUC = {auc_val:.3f})")
    plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate (Recall)")
    plt.title(f"ROC Curve - {model_name}", fontweight="bold")
    plt.legend(loc="lower right")
    roc_path = run_dir / f"roc_{model_name.lower()}.png"
    plt.tight_layout()
    plt.savefig(roc_path, dpi=300)
    plt.close()

    return str(cm_path), str(roc_path)


Appending to src/train.py


In [10]:
%%writefile -a src/train.py


def train_and_evaluate(data_path="data/processed/heart_disease_cleaned.csv", model_dir="models"):
    """Train candidate models, track them in MLflow, and save the champion."""
    Path(model_dir).mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(data_path)
    X = df[ALL_FEATURES]
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("Heart_Disease_Classification_MLOps")

    models_config = {
        "Logistic_Regression": {
            "estimator": LogisticRegression(max_iter=1000, random_state=42),
            "param_grid": {
                "classifier__C": [0.01, 0.1, 1.0, 10.0],
                "classifier__solver": ["lbfgs", "liblinear"],
            },
        },
        "Random_Forest": {
            "estimator": RandomForestClassifier(random_state=42),
            "param_grid": {
                "classifier__n_estimators": [50, 100, 150],
                "classifier__max_depth": [3, 5, 8, None],
                "classifier__min_samples_split": [2, 5],
            },
        },
    }

    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_overall_auc = -1.0
    best_overall_pipeline = None
    best_model_name = ""
    results_summary = []

    for model_name, config in models_config.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", build_preprocessor()),
            ("classifier", config["estimator"]),
        ])

        grid_search = GridSearchCV(
            estimator=pipeline, param_grid=config["param_grid"],
            cv=cv_strategy, scoring="roc_auc", n_jobs=-1, refit=True
        )

        with mlflow.start_run(run_name=model_name) as run:
            run_id = run.info.run_id
            grid_search.fit(X_train, y_train)
            best_pipeline = grid_search.best_estimator_

            cv_scores = cross_validate(
                best_pipeline, X_train, y_train, cv=cv_strategy,
                scoring=["accuracy", "precision", "recall", "roc_auc"]
            )
            y_pred = best_pipeline.predict(X_test)
            y_prob = best_pipeline.predict_proba(X_test)[:, 1]
            roc_score = float(roc_auc_score(y_test, y_prob))

            metrics = {
                "cv_accuracy_mean": float(np.mean(cv_scores["test_accuracy"])),
                "cv_precision_mean": float(np.mean(cv_scores["test_precision"])),
                "cv_recall_mean": float(np.mean(cv_scores["test_recall"])),
                "cv_roc_auc_mean": float(np.mean(cv_scores["test_roc_auc"])),
                "test_accuracy": float(accuracy_score(y_test, y_pred)),
                "test_precision": float(precision_score(y_test, y_pred)),
                "test_recall": float(recall_score(y_test, y_pred)),
                "test_f1": float(f1_score(y_test, y_pred)),
                "test_roc_auc": roc_score,
            }

            mlflow.log_params(grid_search.best_params_)
            mlflow.log_param("cv_folds", 5)
            mlflow.log_param("model_type", model_name)
            mlflow.set_tag("run_id", run_id)
            mlflow.log_metrics(metrics)

            cm_path, roc_path = plot_and_save_artifacts(
                y_test, y_pred, y_prob, model_name, run_id
            )
            mlflow.log_artifact(cm_path, artifact_path="evaluation_plots")
            mlflow.log_artifact(roc_path, artifact_path="evaluation_plots")

            mlflow.sklearn.log_model(
                sk_model=best_pipeline, artifact_path="model", input_example=X_train.iloc[:2]
            )

            historical_dir = Path(model_dir) / "runs"
            historical_dir.mkdir(parents=True, exist_ok=True)
            historical_model_path = historical_dir / f"{model_name}_{run_id}.pkl"
            joblib.dump(best_pipeline, historical_model_path)

            results_summary.append({"model": model_name, "run_id": run_id, **metrics})
            print(f"[{model_name}] Run ID: {run_id}")
            print(f"[{model_name}] Test ROC-AUC: {metrics['test_roc_auc']:.4f}")
            print(f"[{model_name}] Historical model: {historical_model_path}")

            if roc_score > best_overall_auc:
                best_overall_auc = roc_score
                best_overall_pipeline = best_pipeline
                best_model_name = model_name

    final_model_path = Path(model_dir) / "heart_disease_pipeline.pkl"
    joblib.dump(best_overall_pipeline, final_model_path)
    print(f"Champion Model: {best_model_name} (Test ROC-AUC: {best_overall_auc:.4f}) -> Saved to {final_model_path}")
    return pd.DataFrame(results_summary)


if __name__ == "__main__":
    train_and_evaluate()


Appending to src/train.py


In [11]:
import importlib
import src.train

# Reload the generated training module after editing the writefile cells.
importlib.reload(src.train)

summary_df = src.train.train_and_evaluate()
summary_df


/home/s2025ae05884/workspace/heart-disease-predict/venv/lib/python3.12/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/s2025ae05884/workspace/heart-disease-predict/venv/lib/python3.12/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python ca

[Logistic_Regression] Run ID: 5fdbb9ca540541bc885e1a47c5b68987
[Logistic_Regression] Test ROC-AUC: 0.9686
[Logistic_Regression] Historical model: models/runs/Logistic_Regression_5fdbb9ca540541bc885e1a47c5b68987.pkl


/home/s2025ae05884/workspace/heart-disease-predict/venv/lib/python3.12/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/s2025ae05884/workspace/heart-disease-predict/venv/lib/python3.12/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python ca

[Random_Forest] Run ID: 6cd832693bd249b3950be028579b0839
[Random_Forest] Test ROC-AUC: 0.9545
[Random_Forest] Historical model: models/runs/Random_Forest_6cd832693bd249b3950be028579b0839.pkl
Champion Model: Logistic_Regression (Test ROC-AUC: 0.9686) -> Saved to models/heart_disease_pipeline.pkl


,model,run_id,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,Logistic_Regression,5fdbb9ca540541bc885e1a47c5b68987,0.847109,0.876753,0.783399,0.904535,0.885246,0.838710,0.928571,0.881356,0.968615
1,Random_Forest,6cd832693bd249b3950be028579b0839,0.826276,0.859993,0.755336,0.901393,0.901639,0.892857,0.892857,0.892857,0.954545
